In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Oprócz [wizualizacji instrukcji w obwodzie](/guides/visualize-circuits) możesz chcieć zwizualizować harmonogram obwodu za pomocą metody Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer). Ta wizualizacja może pomóc na przykład szybko wykryć czas bezczynności na qubitach. Jednak ta metoda nie zwraca dokładnych wyników dla obwodów dynamicznych. Aby zwizualizować harmonogram obwodów dynamicznych, użyj metody `draw_circuit_schedule_timing`, jak opisano w sekcji [Wsparcie Qiskit Runtime](#qr-support).

## Przykłady

Aby zwizualizować zaplanowany program obwodu, możesz wywołać tę funkcję z zestawem argumentów sterujących. Większość wyglądu obrazu wyjściowego można modyfikować za pomocą arkusza stylów, ale nie jest to wymagane.

### Rysowanie z domyślnym arkuszem stylów

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Rysowanie z arkuszem stylów przystosowanym do debugowania programu

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Możesz tworzyć niestandardowe funkcje generatora lub układu i aktualizować istniejący arkusz stylów za pomocą tych niestandardowych funkcji. W ten sposób możesz kontrolować większość wyglądu obrazu wyjściowego bez modyfikowania bazy kodu rysownika zaplanowanych obwodów. Zobacz dokumentację API [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer), aby uzyskać więcej przykładów.
<span id="qr-support"></span>

## Wsparcie Qiskit Runtime

Chociaż rysownik osi czasu wbudowany w Qiskit jest przydatny dla obwodów statycznych, może nie odzwierciedlać dokładnie taktowania [obwodów dynamicznych](/guides/classical-feedforward-and-control-flow) z powodu operacji niejawnych, takich jak rozgłaszanie i wyznaczanie gałęzi. W ramach wsparcia obwodów dynamicznych Qiskit Runtime zwraca dokładne informacje o taktowaniu obwodu w wynikach zadania, gdy są one żądane.

> **Note:** - Jest to funkcja eksperymentalna. Ma status wersji zapoznawczej i może ulec zmianie.
> - Ta funkcja dotyczy tylko zadań Sampler w Qiskit Runtime.
> - Chociaż całkowity czas obwodu jest zwracany w metadanych "compilation", NIE jest to czas używany do rozliczeń (czas kwantowy).

### Włączanie pobierania danych taktowania

Aby włączyć pobieranie danych taktowania, ustaw eksperymentalną flagę `scheduler_timing` na `True` podczas uruchamiania zadania prymitywu.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Dostęp do danych taktowania obwodu
Gdy są żądane, dane taktowania obwodu dla każdego PUB są zwracane w metadanych wyniku zadania, w `["compilation"]["scheduler_timing"]["timing"]`. To pole zawiera surowe informacje o taktowaniu. Aby wyświetlić informacje o taktowaniu, użyj wbudowanego narzędzia wizualizacji, jak opisano w sekcji [Wizualizacja taktowania](#visualize-timings).

Użyj następującego kodu, aby uzyskać dostęp do danych taktowania obwodu dla pierwszego PUB:

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Zrozumienie surowych danych taktowania
Chociaż wizualizacja danych taktowania obwodu za pomocą metody `draw_circuit_schedule_timing` jest najczęstszym przypadkiem użycia, przydatne może być zrozumienie struktury zwracanych surowych danych taktowania. Może to pomóc na przykład w programowym wyodrębnianiu informacji.

Dane taktowania zwracane w `["compilation"]["scheduler_timing"]["timing"]` to lista ciągów znaków. Każdy ciąg reprezentuje pojedynczą instrukcję na pewnym kanale i jest oddzielony przecinkami na następujące typy danych:

- `Branch` - Określa, czy instrukcja znajduje się w przepływie sterowania (then / else) czy w głównej gałęzi.
- `Instruction` - Bramka i qubit, na którym ma operować.
- `Channel` - Kanał, do którego jest przypisana instrukcja. Może to być jeden z następujących:
      - `Qubit x` - Kanał napędowy dla qubitu _x_.
      - `AWGRx_y` (arbitrary waveform generator readout) - Używany przez kanały odczytu do komunikacji podczas pomiaru qubitów. Argumenty _x_ i _y_ odpowiadają odpowiednio identyfikatorowi instrumentu odczytu i numerowi qubitu.
- `T0` - Czas rozpoczęcia instrukcji w ramach kompletnego harmonogramu
- `Duration` - Czas trwania instrukcji, w jednostkach _dt_ sekund, gdzie 1 dt = 1 cykl planowania. Wartość `dt` backendu możesz znaleźć za pomocą [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Typ wykonywanej operacji impulsowej.

Przykład:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Wizualizacja taktowania
Z `qiskit-ibm-runtime` v0.43.0 lub nowszym możesz wizualizować taktowanie obwodów. Aby zwizualizować taktowanie, musisz najpierw przekonwertować metadane wyniku na `fig` za pomocą [metody `draw_circuit_schedule_timing`.](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) Ta metoda zwraca figurę `plotly`, którą możesz wyświetlić bezpośrednio, zapisać do pliku lub jedno i drugie. Aby uzyskać więcej informacji o poleceniach `plotly`, zobacz [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) i [`fig.write_image("<ścieżka.format>")`.](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![Najechanie na dane wyjściowe pokazuje informacje takie jak start, koniec i czas trwania.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Przykład wygenerowanej figury')
#### Zrozumienie wygenerowanej figury
Obraz danych taktowania obwodu wygenerowany przez `draw_circuit_schedule_timing` przekazuje następujące informacje:

- Oś X to czas w jednostkach _dt_ sekund, gdzie 1 dt = 1 cykl planowania. Wartość `dt` backendu możesz znaleźć za pomocą [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Oś Y to kanał (myśl o kanałach jako o instrumentach emitujących impulsy).
    - `Receive channel` - Jedyny kanał, który sam w sobie nie jest instrumentem. Jest to instrukcja odtwarzana na wszystkich kanałach, które są częścią procedury komunikacji z hubem w danym momencie.
    - `Qubit x` - Kanał napędowy dla qubitu x.
    - `AWGRx_y` (arbitrary waveform generator readout) - Używany przez kanały odczytu do komunikacji podczas pomiaru qubitów. Argumenty _x_ i _y_ odpowiadają odpowiednio identyfikatorowi instrumentu odczytu i numerowi qubitu.
    - `Hub` - Kontroluje rozgłaszanie.

Dodatkowo każda instrukcja ma format *X_Y*, gdzie *X* to nazwa instrukcji, a *Y* to typ impulsu. Typ `play` stosuje impulsy sterujące, a `capture` rejestruje stan qubitu. Możesz również najechać kursorem na każdą instrukcję, aby uzyskać więcej szczegółów. Na przykład poprzednia figura pokazuje impuls sterujący bramki X zastosowany do qubitu 10 przy 1161 dt.
### Przykład od początku do końca
Ten przykład pokazuje, jak włączyć opcję, pobrać informacje o taktowaniu obwodu z metadanych i wyświetlić je jako obraz.

Najpierw skonfiguruj środowisko, zdefiniuj obwody i przekonwertuj je na obwody ISA, a następnie zdefiniuj i uruchom zadania.

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Następnie pobierz taktowanie harmonogramu obwodu: